<a href="https://colab.research.google.com/github/lnittala1-ops/BAN482/blob/main/BAN482_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Training Multi-Layer Perceptrons with Keras and PyTorch


1. **Step 1: Set up Keras** (Configure the `torch` backend and format datasets as numeric tensors)
2. **Step 2: Define the neural network** (Explicitly stack `Input`, hidden `Dense`, and output `Dense` layers using `keras.Sequential`)
3. **Step 3: Train the neural network** (Configure the optimizer and loss function via `.compile()`, then fit via `.fit()`)
4. **Step 4: Evaluate the neural network** (Inspect the training `History` loss curve and compute accuracy via `.evaluate()`)

> **Colab Setup Reminder:** Ensure your runtime is set to your preferred **GPU** via **Runtime > Change runtime type > Hardware accelerator:**.

### Import Libraries, Setup Keras

In [ ]:
# ==============================================================================
# WORKFLOW LINK: Importing Libraries & Setting the Backend
# - In scikit-learn: You import MLPClassifier directly from sklearn.neural_network.
#   scikit-learn runs only on the CPU and handles backend math automatically.
# - In Keras 3: Keras is a high-level API that uses a backend library (PyTorch,
#   TensorFlow, or JAX) to execute tensor operations on a GPU. You MUST set
#   os.environ["KERAS_BACKEND"] = "torch" BEFORE importing keras.
# ==============================================================================
import os
os.environ["KERAS_BACKEND"] = "torch"  # Instructs Keras to use PyTorch for all tensor/GPU operations.

import torch  # Imported to verify GPU availability.
import keras  # High-level deep learning API.
from keras import layers, optimizers  # Modules for building network layers and choosing optimization algorithms.

import numpy as np  # Used for numerical operations and array type conversions.
import pandas as pd  # Used for loading and manipulating CSV data in DataFrames.
import matplotlib.pyplot as plt  # Used for plotting training loss curves.
import seaborn as sns  # High-level interface for statistical graphics.
from sklearn import metrics  # Used for additional model evaluation metrics if needed.

import warnings
warnings.simplefilter("ignore")  # Ignores warnings to keep notebook output clean.

# Verify that Keras is linked to PyTorch and running on the Colab GPU
print("Keras version:", keras.__version__)
print("Active Keras backend:", keras.backend.backend())
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Connected GPU:", torch.cuda.get_device_name(0))

### Data Import

In [ ]:
# Load the Fashion-MNIST training and testing datasets from CSV files.
# (Make sure fashion-mnist_train.csv and fashion-mnist_test.csv are uploaded to your Colab Files sidebar)
train = pd.read_csv("fashion-mnist_train.csv")
test = pd.read_csv("fashion-mnist_test.csv")

print("Training size:", len(train))
print("Testing size:", len(test))

### Modeling Workflow

#### DEFINE

In [ ]:
# ==============================================================================
# WORKFLOW LINK: Data Preparation (DataFrames vs. Typed NumPy Arrays/Tensors)
# - In scikit-learn: MLPClassifier accepts pandas DataFrames directly and
#   internally converts data types.
# - In Keras (PyTorch backend): PyTorch GPU kernels require strict numeric types.
#   Input features (X) must be 32-bit floating point ('float32'), and integer
#   class labels (y) must be 64-bit integers ('int64').
# ==============================================================================

# Extract training features (784 pixel columns) as float32 and target labels (0-9) as int64
Xtrain = train.drop(["label"], axis=1).values.astype("float32")
ytrain = train["label"].values.astype("int64")

# Extract testing features as float32 and target labels as int64
Xtest = test.drop(["label"], axis=1).values.astype("float32")
ytest = test["label"].values.astype("int64")

#### INITIALIZE

In [ ]:
# ==============================================================================
# WORKFLOW LINK: Defining & Training the Multi-Layer Perceptron
#
# Original scikit-learn code:
#   mlpModel = MLPClassifier(random_state=123, learning_rate="constant",
#                            learning_rate_init=0.001, hidden_layer_sizes=(50,),
#                            max_iter=500)
#   mlpModel.fit(Xtrain, ytrain)
#   mlpModel.score(Xtest, ytest)
#
# How each scikit-learn parameter maps to Keras:
# 1. random_state=123         --> keras.utils.set_random_seed(123)
# 2. Input & Output dimensions--> scikit-learn infers 784 inputs and 10 classes
#                                 automatically during .fit(). In Keras, we
#                                 explicitly define layers.Input(shape=(784,))
#                                 and an output layer layers.Dense(10, activation="softmax").
# 3. hidden_layer_sizes=(50,) --> layers.Dense(50, activation="relu")
#                                 (ReLU is the default hidden activation in sklearn).
# 4. solver="adam" (default),
#    learning_rate_init=0.001 --> optimizers.Adam(learning_rate=0.001) inside .compile().
# 5. max_iter=500             --> epochs=500 inside .fit(), paired with
#                                 batch_size=200 (scikit-learn's default batch size).
# ==============================================================================

# Set random seed for reproducibility across NumPy, Python, and PyTorch
keras.utils.set_random_seed(123)

# Define the neural network architecture
mlpModel = keras.Sequential([
    layers.Input(shape=(Xtrain.shape[1],)),       # Input layer accepting 784 pixel features
    layers.Dense(100, activation="relu"), # 1 hidden layer with 100 nodes and ReLU activation
    layers.Dense(10, activation="softmax")        # Output layer with 10 nodes (classes 0-9) and Softmax probabilities
])

# Compile the model (configure optimizer, loss function, and evaluation metric)
# SparseCategoricalCrossentropy is used when integer labels (0-9) are provided instead of one-hot vectors.
mlpModel.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss="SparseCategoricalCrossentropy",
    metrics=["accuracy"]
)

#### TRAIN

In [ ]:
# Train the model on the GPU
# Unlike sklearn, .fit() returns a History object that stores loss and accuracy for each epoch.
history = mlpModel.fit(
    Xtrain, ytrain,
    batch_size=200,  # Matches MLPClassifier's default batch_size=min(200, n_samples)
    epochs=30,
    verbose=1        # Set verbose=1 to watch epoch-by-epoch progress bars
)



#### PREDICT and EVALUATE

In [ ]:
# Evaluate the model on the testing data (Equivalent to mlpModel.score(Xtest, ytest))
# .evaluate() returns [loss, accuracy]
test_loss, test_acc = mlpModel.evaluate(Xtest, ytest, verbose=0)
print("Testing accuracy:", round(test_acc, 4))

In [ ]:
# Evaluate the model's accuracy on the training data (Equivalent to mlpModel.score(Xtrain, ytrain))
train_loss, train_acc = mlpModel.evaluate(Xtrain, ytrain, verbose=0)
print("Training accuracy:", round(train_acc, 4))

In [ ]:
# ==============================================================================
# WORKFLOW LINK: Plotting the Loss Curve
# - In scikit-learn: Loss values are stored as an attribute on the model object
#   itself (mlpModel.loss_curve_).
# - In Keras: Loss and metric values at each epoch are stored in a dictionary
#   inside the History object returned by .fit(): history.history["loss"].
# ==============================================================================

plt.plot(history.history["loss"])  # Plots how cross-entropy loss decreased over 500 epochs.
plt.xlabel("Epochs", fontsize=14)
plt.ylabel("Loss", fontsize=14)
plt.title("MLP Training Loss Curve (Keras - PyTorch)", fontsize=14)
plt.show()

### Comparing Initial Learning Rates

In [ ]:
# ==============================================================================
# WORKFLOW LINK: Exploring Learning Rates
# - In scikit-learn: Calling MLPClassifier(random_state=123, learning_rate_init=lr, max_iter=100)
#   without specifying hidden_layer_sizes defaults to 1 hidden layer of 100 nodes.
# - In Keras: We create a helper function that builds the exact same 100-node
#   architecture and trains it for 100 epochs across each learning rate.
# ==============================================================================

def train_mlp_with_lr(lr):
    keras.utils.set_random_seed(123)  # Equivalent to random_state=123
    model = keras.Sequential([
        layers.Input(shape=(Xtrain.shape[1],)),
        layers.Dense(100, activation="relu"),  # Matches sklearn's default hidden_layer_sizes=(100,)
        layers.Dense(10, activation="softmax")
    ])
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),  # Equivalent to learning_rate_init=lr
        loss="SparseCategoricalCrossentropy",
        metrics=["accuracy"]
    )
    # Train for 100 epochs (max_iter=100) and return the History object
    return model.fit(Xtrain, ytrain, batch_size=200, epochs=30, verbose=0)

# Train four models with different initial learning rates
history0 = train_mlp_with_lr(0.00005)     # Equivalent to mlpModel3 (lr = 0.00005)
history1 = train_mlp_with_lr(0.0001)  # Equivalent to mlpModel0 (lr = 0.0001)
history2 = train_mlp_with_lr(0.0005)    # Equivalent to mlpModel2 (lr = 0.01)
history3 = train_mlp_with_lr(0.001)   # Equivalent to mlpModel1 (lr = 0.001)


print("Finished training all 4 learning rate models on GPU!")

In [ ]:
# Plot loss functions over the first 50 epochs to compare learning rates
# Accessing history.history["loss"][0:50] mirrors mlpModel.loss_curve_[0:50]
plt.plot(history0.history["loss"][0:50], label="0.00005", linestyle="solid")
plt.plot(history1.history["loss"][0:50], label="0.0001", linestyle="dashed")
plt.plot(history2.history["loss"][0:50], label="0.0005", linestyle="dotted")
plt.plot(history3.history["loss"][0:50], label="0.001", linestyle="dashdot")

plt.legend(title="Learning rate")
plt.xlabel("Epochs", fontsize=14)
plt.ylabel("Loss", fontsize=14)
plt.title("Effect of Learning Rate on Convergence", fontsize=14)
plt.show()